In [ ]:
import os
import sys
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from google.colab import drive

# ================================
# 0. Repo: privacy_ml.ppt.bie (İrem Damla — roadmap ile aynı sınıf)
# ================================
if not os.path.exists("/content/Privacy_Preserving_ML"):
    !git clone https://github.com/Erayisci/Privacy_Preserving_ML.git
sys.path.insert(0, "/content/Privacy_Preserving_ML")

from privacy_ml.ppt.bie import BlockWiseImageEncryption

TILE_SIZE = 10
KEY_SEED = 42
_bie = BlockWiseImageEncryption(tile_size=TILE_SIZE, key_seed=KEY_SEED)


def preprocessing_bie(img):
    """Keras ön-işlem: (H,W,C) tek örnek -> BlockWiseImageEncryption.apply."""
    x = np.expand_dims(np.asarray(img, dtype=np.float32), axis=0)
    return _bie.apply(x)[0]


# ================================
# 1. DRIVE BAĞLANTISI VE VERİ TRANSFERİ
# ================================
drive.mount('/content/drive')

drive_path = '/content/drive/MyDrive/chest_xray'
local_path = '/content/chest_xray'

# Verileri yerel diske kopyala (Hızlı eğitim için kritik)
if not os.path.exists(local_path):
    print("Veriler kopyalanıyor, lütfen bekleyin...")
    !cp -r "$drive_path" "$local_path"
    print("Kopyalama tamamlandı.")

train_dir = os.path.join(local_path, "train")
val_dir   = os.path.join(local_path, "val")
test_dir  = os.path.join(local_path, "test")

IMG_SIZE = 150
BATCH_SIZE = 32

# ================================
# 2. DATA GENERATORS — BIE = privacy_ml.ppt.bie.BlockWiseImageEncryption
# ================================
# Egehan runner / encoder_hash: bie_tile_size ve bie_key_seed ile aynı değerleri kullanın.

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    preprocessing_function=preprocessing_bie,
)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    color_mode='grayscale'
)

# Doğrulama (validation) — eğitim sırasında loss/accuracy
val_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    preprocessing_function=preprocessing_bie,
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    color_mode='grayscale',
    shuffle=False,
)

# Test jeneratörü
test_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    preprocessing_function=preprocessing_bie,
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    color_mode='grayscale',
    shuffle=False,
)

# ================================
# 4. CNN MODELİ VE EĞİTİM
# ================================
model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("\nPermuted Model eğitimi başlıyor...")
history = model.fit(
    train_generator,
    epochs=10,
    steps_per_epoch=len(train_generator),
    validation_data=val_generator,
    validation_steps=len(val_generator),
)

# ================================
# 5. SONUÇLARIN GÖRSELLEŞTİRİLMESİ
# ================================
test_loss, test_acc = model.evaluate(test_generator)
print(f"\nPermuted Test Accuracy: {test_acc:.4f}")

# Karıştırılmış bir örnek görüntü göster
sample_img, _ = next(test_generator)
plt.imshow(sample_img[0].reshape(IMG_SIZE, IMG_SIZE), cmap='gray')
plt.title("Block-wise Permuted X-ray Image")
plt.axis('off')
plt.show()